# Optimized Record Linkage: Hybrid Cascade Approach

A high-performance deterministic-first cascade strategy combining exact matching with hybrid fuzzy fallback.

## Key Improvements

1. **Cascade Strategy** — High-confidence exact matches first, then progressively relaxed rules  
2. **Fuzzy as Fallback** — Only apply fuzzy matching to records that didn't match exactly
3. **Hybrid Similarity** — Q-grams for long names, edit distance for short names (handles typos better)
4. **Adaptive Blocking** — Block size limits prevent O(N²) blowup on common dates
5. **Multi-level Thresholds** — Different similarity thresholds per cascade level


In [1]:
import pandas as pd
from collections import defaultdict
from time import time

# Configuration constants for easier tuning
MAX_BLOCK_SIZE_DOB = 100  # Cap fuzzy matching blocks for common birth dates
MAX_BLOCK_SIZE_PLZ = 50   # Cap fuzzy matching blocks for PLZ+year groups
SHORT_NAME_THRESHOLD = 5  # Names shorter than this use edit distance

pd.set_option('display.max_columns', 20)


## 1. Load Data & Preprocess


In [2]:
df = pd.read_csv('../../data/patients_gen.csv', dtype=str)
df['entity_id'] = df['entity_id'].astype(int)

# Vectorized preprocessing for maximum speed
t0 = time()
cols_to_clean = ['EGKVERSICHERTENNUMMER', 'VORNAME', 'NACHNAME', 'Geburtsdatum', 'PLZ']
target_cols = ['_egk', '_vorname', '_nachname', '_geb', '_plz']

for src, tgt in zip(cols_to_clean, target_cols):
    df[tgt] = df[src].fillna('').astype(str).str.upper().str.strip()

print(f"Preprocessing: {len(df):,} records in {time()-t0:.2f}s")
print(f"Unique entities: {df['entity_id'].nunique():,}")
print(f"Records with EGK: {(df['_egk'] != '').sum():,}")
df.head()


Preprocessing: 499,720 records in 0.46s
Unique entities: 100,000
Records with EGK: 357,128


,entity_id,DWH_ZEITRAUM,Datenkoerper,EGKVERSICHERTENNUMMER,VORNAME,NACHNAME,Geburtsdatum,PLZ,GESCHLECHT,ERGEBNISONLINEPRUEFUNG,...,golden_VORNAME,golden_NACHNAME,golden_Geburtsdatum,golden_PLZ,golden_GESCHLECHT,_egk,_vorname,_nachname,_geb,_plz
0,48540,20223,BEARBEITET,X651194502,Bettina,Mohamed,1993-08-03,33105,M,NaN,...,Bettina,Mohamed,1993-08-03,33105,M,X651194502,BETTINA,MOHAMED,1993-08-03,33105
1,458,20201,UNBEARBEITET,G396568870,Markus,Yalcin,1966-01-22,59020,W,1.0,...,Markus,Yalcin,1966-01-22,59020,W,G396568870,MARKUS,YALCIN,1966-01-22,59020
2,3184,20244,UNBEARBEITET,NaN,LAURA,Hoffmann,1998-01-05,59929,M,1.0,...,LAURA,Hoffmann,1998-01-05,45356,M,,LAURA,HOFFMANN,1998-01-05,59929
3,33478,20242,KVUEPP,P205553181,HEIKE,KAMINSKI,1969-11-17,30739,M,1.0,...,HEIKE,KAMINSKI,1969-11-17,30739,M,P205553181,HEIKE,KAMINSKI,1969-11-17,30739
4,6243,20224,KVUEPP,U169787359,BEN,Zimmer,1975-09-21,57835,W,NaN,...,BEN,Zimmer,1975-09-21,57835,W,U169787359,BEN,ZIMMER,1975-09-21,57835


## 2. Similarity Functions

Hybrid strategy:
- **Q-Grams:** For long names (robust to character swaps and OCR errors)
- **Edit Distance:** For short names (prevents over-punishment of single typos in short strings)


In [3]:
def get_qgrams(s: str, q: int = 2) -> set:
    """Extract character q-grams from a string."""
    if len(s) < q:
        return {s} if s else set()
    return {s[i:i+q] for i in range(len(s) - q + 1)}


def qgram_similarity(s1: str, s2: str, q: int = 2) -> float:
    """Jaccard similarity on q-grams. Robust to insertions/deletions."""
    if s1 == s2:
        return 1.0
    if not s1 or not s2:
        return 0.0
    q1, q2 = get_qgrams(s1, q), get_qgrams(s2, q)
    if not q1 or not q2:
        return 0.0
    intersection = len(q1 & q2)
    union = len(q1 | q2)
    return intersection / union if union > 0 else 0.0


def is_distance_at_most_1(s1: str, s2: str) -> bool:
    """O(N) check if Levenshtein distance is <= 1. Fast for short names."""
    if abs(len(s1) - len(s2)) > 1:
        return False
    if s1 == s2:
        return True
    # Substitution check (same length)
    if len(s1) == len(s2):
        diff = sum(1 for c1, c2 in zip(s1, s2) if c1 != c2)
        return diff <= 1
    # Insertion/Deletion check (length diff 1)
    if len(s1) > len(s2):
        s1, s2 = s2, s1  # Ensure s1 is shorter
    for i in range(len(s1)):
        if s1[i] != s2[i]:
            return s1[i:] == s2[i+1:]
    return True


def name_similarity(n1: str, n2: str) -> float:
    """Hybrid: edit distance for short names, q-grams for long names."""
    if n1 == n2:
        return 1.0
    if not n1 or not n2:
        return 0.0
    # Use edit distance for short names (handles typos better)
    if len(n1) < SHORT_NAME_THRESHOLD or len(n2) < SHORT_NAME_THRESHOLD:
        return 0.9 if is_distance_at_most_1(n1, n2) else 0.0
    # Use Q-gram similarity for longer names
    return qgram_similarity(n1, n2, q=2)


# Test
print("Similarity tests:")
print(f"  MUELLER vs MULLER (Q-gram): {qgram_similarity('MUELLER', 'MULLER'):.3f}")
print(f"  SCHMIDT vs SCHMITT (Q-gram): {qgram_similarity('SCHMIDT', 'SCHMITT'):.3f}")
print(f"  FRANK vs FRSNK (hybrid): {name_similarity('FRANK', 'FRSNK'):.3f}")
print(f"  LISA vs LSIA (hybrid): {name_similarity('LISA', 'LSIA'):.3f}")


Similarity tests:
  MUELLER vs MULLER (Q-gram): 0.571
  SCHMIDT vs SCHMITT (Q-gram): 0.500
  FRANK vs FRSNK (hybrid): 0.333
  LISA vs LSIA (hybrid): 0.000


## 3. Union-Find & Cascade Matching Strategy

**Cascade levels** (from highest to lowest confidence):
1. **EGK exact** — Same insurance number = same person
2. **Full name + DOB exact** — Very high confidence
3. **Name fuzzy (≥0.8) + DOB exact** — Handles typos with hybrid similarity
4. **EGK + Last name exact** — Handle first name changes
5. **Name fuzzy + PLZ + birth year** — Lowest level fallback


In [4]:
class UnionFind:
    """Efficient Union-Find with path compression."""
    __slots__ = ['parent', 'rank']
    
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n
    
    def find(self, x: int) -> int:
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x: int, y: int) -> bool:
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True


In [5]:
def cascade_linkage(df: pd.DataFrame) -> pd.Series:
    """
    Multi-level cascade matching: start with highest confidence rules,
    progressively add lower confidence matches.
    """
    n = len(df)
    uf = UnionFind(n)
    
    # Pre-extract arrays for fast access
    egk = df['_egk'].values
    vorname = df['_vorname'].values
    nachname = df['_nachname'].values
    geb = df['_geb'].values
    plz = df['_plz'].values
    
    stats = defaultdict(int)
    
    # === LEVEL 1: EGK exact match ===
    print("Level 1: EGK exact match...")
    t0 = time()
    egk_groups = defaultdict(list)
    for i in range(n):
        if egk[i]:
            egk_groups[egk[i]].append(i)
    
    for indices in egk_groups.values():
        if len(indices) > 1:
            first = indices[0]
            for other in indices[1:]:
                if uf.union(first, other):
                    stats['L1_egk'] += 1
    print(f"  Matches: {stats['L1_egk']:,} ({time()-t0:.1f}s)")
    
    # === LEVEL 2: Full name exact + DOB exact ===
    print("Level 2: Name + DOB exact...")
    t0 = time()
    name_dob_groups = defaultdict(list)
    for i in range(n):
        if vorname[i] and nachname[i] and geb[i]:
            key = (vorname[i], nachname[i], geb[i])
            name_dob_groups[key].append(i)
    
    for indices in name_dob_groups.values():
        if len(indices) > 1:
            first = indices[0]
            for other in indices[1:]:
                if uf.union(first, other):
                    stats['L2_name_dob'] += 1
    print(f"  Matches: {stats['L2_name_dob']:,} ({time()-t0:.1f}s)")
    
    # === LEVEL 3: Name fuzzy + DOB exact ===
    print("Level 3: Name fuzzy + DOB exact...")
    t0 = time()
    # Group by DOB first (exact), then fuzzy match names within group
    dob_groups = defaultdict(list)
    for i in range(n):
        if geb[i] and vorname[i] and nachname[i]:
            dob_groups[geb[i]].append(i)
    
    for dob_val, indices in dob_groups.items():
        # Cap block size to prevent O(N²) blowup on common birth dates
        if len(indices) < 2 or len(indices) > MAX_BLOCK_SIZE_DOB:
            continue
        # Compare pairs within DOB group
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                idx_i, idx_j = indices[i], indices[j]
                if uf.find(idx_i) == uf.find(idx_j):
                    continue
                # Fuzzy name match
                fn_sim = name_similarity(vorname[idx_i], vorname[idx_j])
                ln_sim = name_similarity(nachname[idx_i], nachname[idx_j])
                if fn_sim >= 0.8 and ln_sim >= 0.8:
                    if uf.union(idx_i, idx_j):
                        stats['L3_fuzzy_dob'] += 1
    print(f"  Matches: {stats['L3_fuzzy_dob']:,} ({time()-t0:.1f}s)")
    
    # === LEVEL 4: EGK + Last name exact (handle first name changes) ===
    print("Level 4: EGK + Last name...")
    t0 = time()
    egk_ln_groups = defaultdict(list)
    for i in range(n):
        if egk[i] and nachname[i]:
            key = (egk[i], nachname[i])
            egk_ln_groups[key].append(i)
    
    for indices in egk_ln_groups.values():
        if len(indices) > 1:
            first = indices[0]
            for other in indices[1:]:
                if uf.union(first, other):
                    stats['L4_egk_ln'] += 1
    print(f"  Matches: {stats['L4_egk_ln']:,} ({time()-t0:.1f}s)")
    
    # === LEVEL 5: Name fuzzy + PLZ + year match ===
    print("Level 5: Name fuzzy + PLZ + birth year...")
    t0 = time()
    plz_year_groups = defaultdict(list)
    for i in range(n):
        if plz[i] and geb[i] and len(geb[i]) >= 4 and vorname[i] and nachname[i]:
            key = (plz[i], geb[i][:4])  # PLZ + birth year
            plz_year_groups[key].append(i)
    
    for indices in plz_year_groups.values():
        if len(indices) < 2 or len(indices) > MAX_BLOCK_SIZE_PLZ:
            continue
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                idx_i, idx_j = indices[i], indices[j]
                if uf.find(idx_i) == uf.find(idx_j):
                    continue
                fn_sim = name_similarity(vorname[idx_i], vorname[idx_j])
                ln_sim = name_similarity(nachname[idx_i], nachname[idx_j])
                if fn_sim >= 0.85 and ln_sim >= 0.85:
                    if uf.union(idx_i, idx_j):
                        stats['L5_plz_year'] += 1
    print(f"  Matches: {stats['L5_plz_year']:,} ({time()-t0:.1f}s)")
    
    print(f"\nTotal matches by level: {dict(stats)}")
    
    return pd.Series([uf.find(i) for i in range(n)], index=df.index)


## 4. Run Cascade Linkage


In [6]:
print("=" * 60)
print("IMPROVED RECORD LINKAGE: CASCADE APPROACH")
print("=" * 60 + "\n")

t_start = time()
df['predicted_cluster'] = cascade_linkage(df)
print(f"\nTotal time: {time() - t_start:.1f}s")

print(f"\nResults:")
print(f"  Predicted clusters: {df['predicted_cluster'].nunique():,}")
print(f"  Ground truth entities: {df['entity_id'].nunique():,}")


IMPROVED RECORD LINKAGE: CASCADE APPROACH

Level 1: EGK exact match...
  Matches: 247,497 (0.2s)
Level 2: Name + DOB exact...
  Matches: 98,834 (0.4s)
Level 3: Name fuzzy + DOB exact...
  Matches: 5,139 (7.3s)
Level 4: EGK + Last name...
  Matches: 0 (0.3s)
Level 5: Name fuzzy + PLZ + birth year...
  Matches: 720 (0.9s)

Total matches by level: {'L1_egk': 247497, 'L2_name_dob': 98834, 'L3_fuzzy_dob': 5139, 'L4_egk_ln': 0, 'L5_plz_year': 720}

Total time: 9.4s

Results:
  Predicted clusters: 147,530
  Ground truth entities: 100,000


## 5. Evaluation


In [7]:
def count_pairs(labels: pd.Series) -> int:
    """Count pairs within clusters."""
    counts = labels.value_counts()
    return int((counts * (counts - 1) / 2).sum())


def compute_metrics(true_labels: pd.Series, pred_labels: pd.Series) -> dict:
    """Compute pairwise precision, recall, F1."""
    # Group by predicted cluster
    pred_clusters = defaultdict(list)
    for i, pid in enumerate(pred_labels):
        pred_clusters[pid].append(i)
    
    # Count true positives
    tp = 0
    for members in pred_clusters.values():
        true_counts = defaultdict(int)
        for idx in members:
            true_counts[true_labels.iloc[idx]] += 1
        for c in true_counts.values():
            tp += c * (c - 1) // 2
    
    pred_pairs = count_pairs(pred_labels)
    true_pairs = count_pairs(true_labels)
    
    precision = tp / pred_pairs if pred_pairs > 0 else 0
    recall = tp / true_pairs if true_pairs > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    
    return {'tp': tp, 'pred_pairs': pred_pairs, 'true_pairs': true_pairs,
            'precision': precision, 'recall': recall, 'f1': f1}


def analyze_clusters(df: pd.DataFrame) -> dict:
    """Analyze cluster quality."""
    results = {'perfect': 0, 'split': 0, 'merged': 0}
    
    for pred_id, group in df.groupby('predicted_cluster'):
        true_ids = group['entity_id'].unique()
        if len(true_ids) == 1:
            entity_id = true_ids[0]
            if len(group) == (df['entity_id'] == entity_id).sum():
                results['perfect'] += 1
            else:
                results['split'] += 1
        else:
            results['merged'] += 1
    
    return results


In [8]:
metrics = compute_metrics(df['entity_id'], df['predicted_cluster'])
clusters = analyze_clusters(df)
total_clusters = df['predicted_cluster'].nunique()

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

print(f"\nPairwise Metrics:")
print(f"  True pairs:      {metrics['true_pairs']:,}")
print(f"  Predicted pairs: {metrics['pred_pairs']:,}")
print(f"  True positives:  {metrics['tp']:,}")
print(f"\n  Precision: {metrics['precision']*100:.2f}%")
print(f"  Recall:    {metrics['recall']*100:.2f}%")
print(f"  F1 Score:  {metrics['f1']*100:.2f}%")

print(f"\nCluster Analysis:")
print(f"  Total clusters: {total_clusters:,}")
print(f"  Perfect:  {clusters['perfect']:,} ({clusters['perfect']/total_clusters*100:.1f}%)")
print(f"  Split:    {clusters['split']:,} ({clusters['split']/total_clusters*100:.1f}%)")
print(f"  Merged:   {clusters['merged']:,} ({clusters['merged']/total_clusters*100:.1f}%)")


EVALUATION RESULTS

Pairwise Metrics:
  True pairs:      1,199,760
  Predicted pairs: 984,039
  True positives:  983,177

  Precision: 99.91%
  Recall:    81.95%
  F1 Score:  90.04%

Cluster Analysis:
  Total clusters: 147,530
  Perfect:  61,804 (41.9%)
  Split:    85,664 (58.1%)
  Merged:   62 (0.0%)


## 6. Error Analysis


In [9]:
cols = ['entity_id', 'predicted_cluster', 'EGKVERSICHERTENNUMMER', 'VORNAME', 'NACHNAME', 'Geburtsdatum', 'PLZ']

# False Positives
print("=" * 80)
print("FALSE POSITIVES (Different entities merged)")
print("=" * 80)
fp_count = 0
for pred_id, group in df.groupby('predicted_cluster'):
    if len(group['entity_id'].unique()) > 1:
        if fp_count < 3:
            display(group[cols].head(6))
        fp_count += 1
print(f"\nTotal merged clusters (FP): {fp_count}")


FALSE POSITIVES (Different entities merged)


,entity_id,predicted_cluster,EGKVERSICHERTENNUMMER,VORNAME,NACHNAME,Geburtsdatum,PLZ
3700,69969,3700,H711166212,S.,KÖNIG,1985-06-28,37947
152548,20049,3700,G255314150,INGE,KÖNIG,1985-06-28,59523
161763,20049,3700,G255314150,I.,KÖNIG,1985-06-28,59523
174518,20049,3700,G255314150,inge,KÖNIG,1985-06-28,59523
177991,20049,3700,G255314150,INGE,KÖNIG,1985-06-28,59523
184279,69969,3700,H711166212,SVETLANA,KÖNIG,1985-06-28,37947


,entity_id,predicted_cluster,EGKVERSICHERTENNUMMER,VORNAME,NACHNAME,Geburtsdatum,PLZ
4679,44039,4679,N003898928,MATTHIAS,Schmitz,1985-06-12,46296
45510,17316,4679,Y799272837,MATTHIAS,SCHMITZ,1985-06-12,64976
132342,17316,4679,Y799272837,matthias,SCHMITZ,1985-06-12,64976
184209,44039,4679,N003898928,MATTHIAS,Schmitz,1985-06-12,46296
189601,17316,4679,Y799272837,MATTHIAS,SCHMITZ,1985-06-12,64976
285832,17316,4679,NaN,MATTHIAS,SCHMITZ,1985-06-12,32031


,entity_id,predicted_cluster,EGKVERSICHERTENNUMMER,VORNAME,NACHNAME,Geburtsdatum,PLZ
6683,52524,6683,A915058327,ANNA,BAUER,1960-08-20,48904
110586,81919,6683,V375337510,ANNA,LUDWIG,1960-08-20,58222
230943,81919,6683,V375337510,ANNA,LUDWIG,1960-08-20,58222
250347,52524,6683,A915058327,ANNA,BAUER,1960-08-20,48904
404853,52524,6683,A915058327,ANNA,B.,1960-08-20,48904
461779,81919,6683,V375337510,ANNA,L.,1960-08-20,58222



Total merged clusters (FP): 62


In [10]:
# False Negatives  
print("=" * 80)
print("FALSE NEGATIVES (Same entity split)")
print("=" * 80)
fn_count = 0
for entity_id, group in df.groupby('entity_id'):
    if len(group['predicted_cluster'].unique()) > 1:
        if fn_count < 3:
            display(group[cols].head(6))
        fn_count += 1
print(f"\nTotal split entities (FN): {fn_count}")


FALSE NEGATIVES (Same entity split)


,entity_id,predicted_cluster,EGKVERSICHERTENNUMMER,VORNAME,NACHNAME,Geburtsdatum,PLZ
31716,0,31716,NaN,.,Göbel,1951-02-02,48511
64142,0,64142,Q940265422,EWALD,Göbel,1951-02-02,48511
140300,0,64142,Q940265422,EWALD,Göbel,1951-02-02,48511
195591,0,64142,NaN,EWALD,Göbel,1951-02-02,NaN
305029,0,64142,Q940265422,EWALD,Göbel,1951-02-02,48511


,entity_id,predicted_cluster,EGKVERSICHERTENNUMMER,VORNAME,NACHNAME,Geburtsdatum,PLZ
32348,1,32348,NaN,HEIKE,Schäfer,1982-06-09,42146
138627,1,32348,I784801842,HEIKE,Schäfer,1982-06-09,42146
156897,1,156897,NaN,HEIK.,Schä.,1982-06-09,42146


,entity_id,predicted_cluster,EGKVERSICHERTENNUMMER,VORNAME,NACHNAME,Geburtsdatum,PLZ
16753,3,16753,U401640050,A.,NaN,2010-03-20,25838
46604,3,46604,U751079910,NaN,NOLTE,2010-03-20,25838
399402,3,46604,U751079910,ANDREA,NOLTE,2010-03-20,25838
418791,3,46604,U751079910,ANDREA,NOLTE,2010-03-20,25838



Total split entities (FN): 38139


## 7. Summary


In [11]:
print("\n" + "=" * 70)
print("CASCADE APPROACH SUMMARY")
print("=" * 70)

print(f"""
Dataset:
  Records: {len(df):,}
  Ground truth entities: {df['entity_id'].nunique():,}
  Predicted clusters: {df['predicted_cluster'].nunique():,}

Metrics:
  Precision: {metrics['precision']*100:.2f}%
  Recall:    {metrics['recall']*100:.2f}%
  F1 Score:  {metrics['f1']*100:.2f}%

Cascade Levels:
  L1: EGK exact match
  L2: Full name + DOB exact  
  L3: Name fuzzy (≥0.8) + DOB exact (hybrid similarity)
  L4: EGK + Last name exact
  L5: Name fuzzy + PLZ + birth year

Key Improvements:
  ✓ Prioritizes high-confidence exact matches first
  ✓ Hybrid similarity: edit distance for short names, Q-grams for long
  ✓ Blocking caps prevent O(N²) blowup on common dates
  ✓ Efficient cascade prevents over-merging
""")
print("=" * 70)



CASCADE APPROACH SUMMARY

Dataset:
  Records: 499,720
  Ground truth entities: 100,000
  Predicted clusters: 147,530

Metrics:
  Precision: 99.91%
  Recall:    81.95%
  F1 Score:  90.04%

Cascade Levels:
  L1: EGK exact match
  L2: Full name + DOB exact  
  L3: Name fuzzy (≥0.8) + DOB exact (hybrid similarity)
  L4: EGK + Last name exact
  L5: Name fuzzy + PLZ + birth year

Key Improvements:
  ✓ Prioritizes high-confidence exact matches first
  ✓ Hybrid similarity: edit distance for short names, Q-grams for long
  ✓ Blocking caps prevent O(N²) blowup on common dates
  ✓ Efficient cascade prevents over-merging

